In [3]:
import pandas as pd
import numpy as np
print("Libraries loaded successfully!")

Libraries loaded successfully!


In [4]:
df = pd.read_csv('merged_cleaned_scaled.csv')
df['DATE_TIME'] = pd.to_datetime(df['DATE_TIME'])
print("Loaded shape:", df.shape)

Loaded shape: (136476, 14)


In [5]:
# Each of the 44 inverters is its own independent time series,
# so we group by (PLANT_ID, SOURCE_KEY) before sorting by time
df = df.sort_values(['PLANT_ID', 'SOURCE_KEY', 'DATE_TIME']).reset_index(drop=True)

In [6]:
# Team decision: only 3 environmental inputs, AC_POWER excluded from inputs
feature_cols = ['IRRADIATION_scaled', 'AMBIENT_TEMPERATURE_scaled', 'MODULE_TEMPERATURE_scaled']
target_col = 'AC_POWER_scaled'

In [7]:
# Split by calendar time, not randomly, to avoid the model "seeing the future"
min_date, max_date = df['DATE_TIME'].min(), df['DATE_TIME'].max()
total_seconds = (max_date - min_date).total_seconds()
train_end = min_date + pd.Timedelta(seconds=total_seconds * 0.70)
val_end   = min_date + pd.Timedelta(seconds=total_seconds * 0.85)

print("Train ends:", train_end)
print("Validation ends:", val_end)

Train ends: 2020-06-07 19:01:29.999999999
Validation ends: 2020-06-12 21:23:15


In [8]:
def build_windows(window_size):
    X_train, y_train = [], []
    X_val,   y_val   = [], []
    X_test,  y_test  = [], []

    # Process each inverter separately so windows never mix different machines
    for (plant, inverter), group in df.groupby(['PLANT_ID', 'SOURCE_KEY']):
        group = group.reset_index(drop=True)
        feats = group[feature_cols].values
        target = group[target_col].values
        times = pd.to_datetime(group['DATE_TIME'].values)

        # True where consecutive timestamps are exactly 15 minutes apart
        is_continuous = (np.diff(times) == np.timedelta64(15, 'm'))

        n = len(group)
        for i in range(n - window_size):
            # Skip windows that cross a real data gap
            if not is_continuous[i:i + window_size].all():
                continue

            X_seq = feats[i:i + window_size]
            y_next = target[i + window_size]
            t_target = times[i + window_size]

            if t_target < train_end:
                X_train.append(X_seq); y_train.append(y_next)
            elif t_target < val_end:
                X_val.append(X_seq); y_val.append(y_next)
            else:
                X_test.append(X_seq); y_test.append(y_next)

    return (np.array(X_train), np.array(y_train),
            np.array(X_val),   np.array(y_val),
            np.array(X_test),  np.array(y_test))

In [9]:
for window_size in [24, 32, 48]:
    X_train, y_train, X_val, y_val, X_test, y_test = build_windows(window_size)

    print(f"\n--- Window size {window_size} ---")
    print("Train:", X_train.shape, y_train.shape)
    print("Val:  ", X_val.shape, y_val.shape)
    print("Test: ", X_test.shape, y_test.shape)

    np.save(f'X_train_w{window_size}.npy', X_train)
    np.save(f'y_train_w{window_size}.npy', y_train)
    np.save(f'X_val_w{window_size}.npy',   X_val)
    np.save(f'y_val_w{window_size}.npy',   y_val)
    np.save(f'X_test_w{window_size}.npy',  X_test)
    np.save(f'y_test_w{window_size}.npy',  y_test)

print("\nAll 18 files saved.")


--- Window size 24 ---
Train: (82361, 24, 3) (82361,)
Val:   (21516, 24, 3) (21516,)
Test:  (20988, 24, 3) (20988,)

--- Window size 32 ---
Train: (78957, 32, 3) (78957,)
Val:   (21516, 32, 3) (21516,)
Test:  (20812, 32, 3) (20812,)

--- Window size 48 ---
Train: (72407, 48, 3) (72407,)
Val:   (21516, 48, 3) (21516,)
Test:  (20460, 48, 3) (20460,)

All 18 files saved.


In [10]:
for window_size in [24, 32, 48]:
    for split in ['train', 'val', 'test']:
        files.download(f'X_{split}_w{window_size}.npy')
        files.download(f'y_{split}_w{window_size}.npy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# Confirm shapes make sense: (num_windows, window_size, 3 features)
for window_size in [24, 32, 48]:
    X_test = np.load(f'X_test_w{window_size}.npy')
    print(f"window={window_size} -> X_test shape:", X_test.shape)

window=24 -> X_test shape: (20988, 24, 3)
window=32 -> X_test shape: (20812, 32, 3)
window=48 -> X_test shape: (20460, 48, 3)
